# Notebook 12 — Phase 2: GitHub Actions Continuous Integration (CI)




In [1]:
# Cell 1 — Find the project root and check Git status
from pathlib import Path
import subprocess
import shutil

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "notebooks").exists() and (PROJECT_ROOT.parent / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project root:", PROJECT_ROOT)
if shutil.which("git") is None:
    raise EnvironmentError("Git is not installed or not available on PATH.")

status = subprocess.run(
    ["git", "status", "--short", "--branch"],
    cwd=PROJECT_ROOT, capture_output=True, text=True
)
print(status.stdout)
if status.returncode != 0:
    print(status.stderr)
    raise RuntimeError("Git status failed. Resolve the Git issue before configuring CI.")


Project root: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system
## main...origin/main [ahead 1]
 M notebooks/11_phase2_git_setup.ipynb
?? notebooks/12_phase2_github_actions_ci.ipynb
?? src/



In [2]:
# Cell 2 — Create a synthetic model-artifact builder for CI
# This script creates a small test-only estimator and metadata.
# It does not train or replace your real fraud-detection model.
from pathlib import Path

ci_script = r'''
from pathlib import Path
import json
import joblib
import pandas as pd
from sklearn.dummy import DummyClassifier

ROOT = Path(__file__).resolve().parents[1]
MODELS = ROOT / "models"
MODELS.mkdir(parents=True, exist_ok=True)

features = [
    "type", "amount", "oldbalanceOrg", "oldbalanceDest", "hour_of_day",
    "night_flag", "amount_to_origin_balance_ratio", "origin_balance_was_zero",
    "origin_txn_count_prior_steps"
]
X = pd.DataFrame([
    ["TRANSFER", 100.0, 200.0, 50.0, 10, 0, 0.5, 0, 1],
    ["CASH_OUT", 250.0, 300.0, 0.0, 23, 1, 0.8, 0, 2],
], columns=features)
y = [0, 1]

model = DummyClassifier(strategy="prior")
model.fit(X, y)
joblib.dump(model, MODELS / "fraud_detection_pipeline.pkl")
metadata = {
    "features": features,
    "feature_names": features,
    "threshold": 0.5,
    "artifact_note": "SYNTHETIC CI TEST MODEL ONLY — never use for real predictions."
}
(MODELS / "fraud_detection_metadata.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
print("Created synthetic CI-only model artifacts.")
'''
script_path = PROJECT_ROOT / "tests" / "create_ci_model.py"
script_path.parent.mkdir(parents=True, exist_ok=True)
script_path.write_text(ci_script.strip() + "\n", encoding="utf-8")
print("Created:", script_path)


Created: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\tests\create_ci_model.py


In [3]:
# Cell 3 — Create the GitHub Actions workflow
workflow = r'''
name: Fraud Detection CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]
  workflow_dispatch:

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - name: Check out repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: "pip"

      - name: Install test dependencies
        run: |
          python -m pip install --upgrade pip
          python -m pip install fastapi uvicorn httpx pytest joblib scikit-learn pandas numpy

      - name: Create synthetic model for tests only
        run: python tests/create_ci_model.py

      - name: Run pytest
        run: python -m pytest -v
'''
workflow_dir = PROJECT_ROOT / ".github" / "workflows"
workflow_dir.mkdir(parents=True, exist_ok=True)
workflow_path = workflow_dir / "ci.yml"
workflow_path.write_text(workflow.strip() + "\n", encoding="utf-8")
print("Created:", workflow_path)
print(workflow_path.read_text(encoding="utf-8"))


Created: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\.github\workflows\ci.yml
name: Fraud Detection CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]
  workflow_dispatch:

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - name: Check out repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: "pip"

      - name: Install test dependencies
        run: |
          python -m pip install --upgrade pip
          python -m pip install fastapi uvicorn httpx pytest joblib scikit-learn pandas numpy

      - name: Create synthetic model for tests only
        run: python tests/create_ci_model.py

      - name: Run pytest
        run: python -m pytest -v



In [4]:
# Cell 4 — Verify CI files without modifying your real local model
workflow_path = PROJECT_ROOT / ".github" / "workflows" / "ci.yml"
script_path = PROJECT_ROOT / "tests" / "create_ci_model.py"

assert workflow_path.exists(), "CI workflow file was not created."
assert script_path.exists(), "CI synthetic-model builder was not created."

print("Workflow exists:", workflow_path.exists())
print("Synthetic CI model builder exists:", script_path.exists())
print("\nThe synthetic model is created only on GitHub Actions runners.")
print("Your real local model artifact has not been changed.")


Workflow exists: True
Synthetic CI model builder exists: True

The synthetic model is created only on GitHub Actions runners.
Your real local model artifact has not been changed.


In [5]:
# Cell 5 — Run the existing local pytest suite using your current local model
# This does not generate or overwrite model artifacts.
import sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v"],
    cwd=PROJECT_ROOT, capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)
print("pytest exit code:", result.returncode)
if result.returncode != 0:
    raise RuntimeError("Local pytest run failed. Review the output before pushing.")


============================= test session starts =============================
platform win32 -- Python 3.14.8, pytest-9.1.1, pluggy-1.6.0 -- c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\Tanish.Ranjan\OneDrive - ENCORA\Desktop\fraud-detection-system
configfile: pytest.ini
testpaths: tests
plugins: anyio-4.15.1, platformdirs-4.13.0
collecting ... collected 6 items

tests/test_api.py::test_health_endpoint PASSED                           [ 16%]
tests/test_api.py::test_model_info_endpoint PASSED                       [ 33%]
tests/test_api.py::test_valid_prediction_request PASSED                  [ 50%]
tests/test_api.py::test_negative_amount_is_rejected PASSED               [ 66%]
tests/test_api.py::test_unexpected_field_is_rejected PASSED              [ 83%]
tests/test_api.py::test_invalid_hour_is_rejected PASSED                  [100%]

============================== warnings summary =========

In [6]:
# Cell 6 — Check the files that should be reviewed for commit
status = subprocess.run(
    ["git", "status", "--short"],
    cwd=PROJECT_ROOT, capture_output=True, text=True
)
print(status.stdout)
print("Review these paths before committing:")
print(" - .github/workflows/ci.yml")
print(" - tests/create_ci_model.py")
print(" - tests/test_api.py")
print(" - app.py")
print(" - pytest.ini")
print(" - notebooks (if you intend to version them)")
print("\nDo not commit .venv, datasets, credentials, or real model pickle files.")


 M notebooks/11_phase2_git_setup.ipynb
?? .github/
?? notebooks/12_phase2_github_actions_ci.ipynb
?? src/
?? tests/create_ci_model.py

Review these paths before committing:
 - .github/workflows/ci.yml
 - tests/create_ci_model.py
 - tests/test_api.py
 - app.py
 - pytest.ini
 - notebooks (if you intend to version them)

Do not commit .venv, datasets, credentials, or real model pickle files.
